In [2]:
import pandas as pd
import numpy as np
import re
import os
import glob
from Bio import SeqIO
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [3]:
cds_gff = "/mnt/SSD7TB/PROJECTS/AKKM835_ncbi/AKKM_835_RefSeq_genomic_cds_PARCED.csv"
interpro_res = "/mnt/SSD7TB/PROJECTS/AKKM835_ncbi/Interpro_out/AKKM_835.faa.tsv"

In [4]:
cds_gff_df = pd.read_csv(cds_gff,sep=",")
interpro_df = pd.read_csv(interpro_res,sep="\t",header=None)

In [16]:
print(cds_gff_df.shape[0])
cds_gff_df.head(3)

2151


,seqid,source,start,end,strand,Parent,locus_tag,product,partial,pseudo,protein_id,go_process,go_function,Note
0,NZ_CP071807.1,Protein Homology,100,1506,+,gene-pgaptmp_000001,pgaptmp_000001,chromosomal replication initiator protein DnaA,NaN,NaN,extdb:pgaptmp_000001,"DNA replication initiation|0006270||IEA,regula...","DNA binding|0003677||IEA,DNA replication origi...",NaN
1,NZ_CP071807.1,Protein Homology,1656,2591,+,gene-pgaptmp_000002,pgaptmp_000002,ABC transporter ATP-binding protein,NaN,NaN,extdb:pgaptmp_000002,NaN,ATP binding|0005524||IEA,NaN
2,NZ_CP071807.1,Protein Homology,2647,3429,+,gene-pgaptmp_000003,pgaptmp_000003,ABC transporter permease subunit,NaN,NaN,extdb:pgaptmp_000003,transmembrane transport|0055085||IEA,ABC-type transporter activity|0140359||IEA,NaN


In [13]:
print(interpro_df.shape[0])
interpro_df.head(3)

16173


,0,annot_DB,DB_domain_ID,DB_domain_annot,evalue,InterPro_domain,InterPro_annot,locus_tag,total_length
0,gnl|extdb|pgaptmp_001480,PANTHER,PTHR35024,HYPOTHETICAL CYTOSOLIC PROTEIN,5.0E-12,IPR007607,Bactofilin A/B,pgaptmp_001480,114
1,gnl|extdb|pgaptmp_001480,Pfam,PF04519,Polymer-forming cytoskeletal,9.0E-8,IPR007607,Bactofilin A/B,pgaptmp_001480,106
2,gnl|extdb|pgaptmp_001480,Pfam,PF04519,Polymer-forming cytoskeletal,0.0056,IPR007607,Bactofilin A/B,pgaptmp_001480,106


In [7]:
interpro_df['locus_tag'] = interpro_df[0].str.extract(r'extdb\|(.+)', expand=False)

In [8]:
interpro_df.drop([1, 2, 6, 7, 9, 10, 13, 14], axis=1, inplace=True)
interpro_df.rename(columns={
    3: 'annot_DB',
    4: 'DB_domain_ID',
    5: 'DB_domain_annot',
    8: 'evalue',
    11: 'InterPro_domain',
    12: 'InterPro_annot'
}, inplace=True)

In [9]:
print(f" Значения NaN в названии белка {interpro_df['locus_tag'].isna().sum()}")

 Значения NaN в названии белка 0


In [10]:
missing_pgaptmp = set(cds_gff_df['locus_tag']) - set(interpro_df['locus_tag'])
print(f"pgaptmps in cds_gff_df not found in interpro_df: {len(missing_pgaptmp)}")

pgaptmps in cds_gff_df not found in interpro_df: 244


In [11]:
print(cds_gff_df['locus_tag'].nunique()) # Количество CDS
print(interpro_df['locus_tag'].nunique()) # Количество CDS, для которых Interpro нашёл домены
print(len(missing_pgaptmp)) # Количество CDS в cds_gff_df, но отсутсвующих в InterPro - не найдены для них домены, в основном это hyp.prot

2151
1907
244


In [12]:
interpro_df['total_length'] = interpro_df.apply(lambda row: sum(len(str(x)) for x in row), axis=1)

# Сортируем по убыванию длины и оставляем первую строку для каждого locus_tag
interpro_res = interpro_df.sort_values('total_length', ascending=False)
print(interpro_res.shape[0])
interpro_res.drop_duplicates('locus_tag', keep='first',inplace=True)
print(interpro_res.shape[0])

16173
1907


In [14]:
cds_gff_df.columns

Index(['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase',
       'attributes', 'ID', 'Name', 'gbkey', 'Parent', 'locus_tag', 'product',
       'partial', 'pseudo', 'protein_id', 'go_process', 'go_function', 'Note'],
      dtype='object')

In [15]:
cds_gff_df.drop(['type','score','phase','attributes', 'ID', 'Name', 'gbkey'], axis=1, inplace=True)

In [26]:
cds_interpro = pd.merge(cds_gff_df, interpro_res, on=['locus_tag'], how='outer')

In [18]:
cds_interpro.to_csv('/mnt/SSD7TB/PROJECTS/AKKM835_ncbi/AKKM_835_CDS_gff_Interpro.csv', sep=",", index=False)

In [27]:
print(f"Кодирующих рамок в регионе 50кб, в которые входит профаг: {cds_interpro[(cds_interpro['start']>=1609000)&(cds_interpro['end']<=1650000)&(cds_interpro['annot_DB'].notna())].shape[0]}")
cds_interpro.drop([0, 'total_length'], axis=1, inplace=True)
prophage_cds_interpro = cds_interpro[(cds_interpro['start']>=1609000)&(cds_interpro['end']<=1650000)&(cds_interpro['annot_DB'].notna())]

Кодирующих рамок в регионе 50кб, в которые входит профаг: 29


In [28]:
prophage_cds_interpro.to_csv('/mnt/SSD7TB/PROJECTS/AKKM835_ncbi/AKKM_835_prophage_cds_interpro.csv', sep=",", index=False)